# 🧠 NLP Text Preprocessing — Complete Guide
### Stemming, Lemmatization, Stop Words, Regex & the Full Pipeline

---

## 📌 The Standard Preprocessing Sequence

When raw text comes in, we run it through a **pipeline** in this order:

```
1. Lowercase
2. Remove URLs / Emails
3. Remove HTML Tags
4. Remove Special Characters & Punctuation
5. Normalize Whitespace
6. Tokenization
7. Stop Word Removal
8. Stemming  OR  Lemmatization
```

> **Why this order?** Each step cleans the output for the next step. Lowercasing first ensures consistent matching. HTML removal before punctuation removal prevents regex collisions. Tokenization must come before stop word removal and stemming/lemmatization.

---
## ⚙️ Setup — Install & Import Everything

In [ ]:
# Run this cell first
import subprocess
subprocess.run(['pip', 'install', 'nltk', 'beautifulsoup4'], capture_output=True)

import re
import nltk

# Download required NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

print("✅ All libraries ready!")

---
## Step 1 — Lowercasing

"Python", "PYTHON", "python" — a computer sees these as 3 different words. We fix that by converting everything to lowercase.

> ⚠️ **Nuance:** Named entities like `Apple` (company) vs `apple` (fruit) get confused. For general NLP, we accept this tradeoff.

In [ ]:
text = "The QUICK Brown Fox Is RUNNING Swiftly"

text_lower = text.lower()
print("Before:", text)
print("After: ", text_lower)

---
## Step 2 — Remove URLs and Emails

URLs and emails are noise — they add nothing to meaning and create garbage tokens in your vocabulary.

In [ ]:
text = "Visit https://example.com or www.google.com — email me at hello@test.com"

# Remove URLs
text = re.sub(r'http\S+|www\S+', '', text)

# Remove emails
text = re.sub(r'\S+@\S+', '', text)

print("Cleaned:", text)

# --- Pattern Breakdown ---
# http\S+  →  'http' followed by any non-whitespace characters (\S+)
# www\S+   →  'www' followed by any non-whitespace characters
# |        →  OR (match either pattern)
# \S+@\S+  →  non-whitespace chars, then @, then more non-whitespace = email

---
## Step 3 — Remove HTML Tags

Text scraped from websites contains HTML markup. Two ways to strip it:

In [ ]:
html_text = "<p>The <b>Quick</b> Brown <a href='#'>Fox</a> is running!</p>"

# Method 1: Regex (fast but dumb)
clean_regex = re.sub(r'<[^>]+>', '', html_text)
print("Regex method:      ", clean_regex)

# Method 2: BeautifulSoup (slower but smarter)
from bs4 import BeautifulSoup
clean_bs = BeautifulSoup(html_text, 'html.parser').get_text()
print("BeautifulSoup method:", clean_bs)

# --- Pattern Breakdown: <[^>]+> ---
# <      →  literal < character
# [^>]   →  any character that is NOT >
# +      →  one or more of those characters
# >      →  literal > character
# Together: matches anything from < to > — that's an HTML tag

---
## Step 4 — Remove Special Characters & Punctuation

Punctuation adds noise for keyword tasks.

> ⚠️ **Don't always do this blindly:**
> - In **legal text** — `Section 4.2.1` matters
> - In **sentiment analysis** — `!!!` expresses intensity
> - In **code analysis** — `!=` means something

In [ ]:
text = "Hello, World!!! This is NLP — it's amazing (really)."

# Keep only letters, numbers, spaces
clean1 = re.sub(r'[^a-zA-Z0-9\s]', '', text)
print("Keep alphanumeric:", clean1)

# Keep only letters and spaces (remove numbers too)
clean2 = re.sub(r'[^a-zA-Z\s]', '', text)
print("Keep letters only:", clean2)

# --- Pattern Breakdown: [^a-zA-Z0-9\s] ---
# [...]    →  character class (match any ONE character inside)
# ^        →  when INSIDE [], means NOT
# a-zA-Z   →  any letter (upper or lowercase)
# 0-9      →  any digit
# \s       →  any whitespace
# So [^a-zA-Z0-9\s] = anything that is NOT a letter, digit, or space

---
## Step 5 — Whitespace Normalization

After cleaning, you end up with double spaces, tabs, newlines. Collapse them all into single spaces.

In [ ]:
text = "hello    world  \t\t  how   \n are   you"

print("Before:", repr(text))  # repr shows the hidden characters

clean = re.sub(r'\s+', ' ', text).strip()
print("After: ", repr(clean))

# --- Pattern Breakdown: \s+ ---
# \s   →  any whitespace character (space, tab \t, newline \n, carriage return \r)
# +    →  one or more of them
# Replace ALL consecutive whitespace with a single space
# .strip() removes leading/trailing spaces

---
## Step 6 — Tokenization ⭐

The most important step conceptually. You **split text into individual units** called tokens.

This is the bridge between raw text and numerical processing.

> 📝 **Note:** LLMs like GPT have their OWN internal tokenizers (BPE, WordPiece). What we do here is NLP-level preprocessing tokenization — different thing entirely.

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize

text = "I love NLP! Don't you think it's powerful?"

# Simple split — dumb, misses edge cases
simple = text.split()
print("Simple split:    ", simple)

# NLTK word tokenizer — handles contractions, punctuation
tokens = word_tokenize(text)
print("NLTK word tokens:", tokens)

# Sentence tokenizer
text2 = "I love NLP. It's powerful. You should try it!"
sentences = sent_tokenize(text2)
print("\nSentences:")
for i, s in enumerate(sentences):
    print(f"  {i+1}: {s}")

# Notice: "Don't" → ["Do", "n't"] — NLTK is smarter than split()

---
## Step 7 — Stop Word Removal

Stop words are extremely common words with almost no meaning — "the", "is", "a", "and", "in", "of".

| ✅ Remove stop words for | ❌ Do NOT remove for |
|---|---|
| TF-IDF, BM25, keyword search | BERT, GPT, transformer models |
| Fast retrieval systems | Sentiment analysis |
| Search engines | Grammar-sensitive tasks |

In [ ]:
from nltk.corpus import stopwords

# See what's in the stop words list
stop_words = set(stopwords.words('english'))
print(f"Total English stop words: {len(stop_words)}")
print("Sample:", sorted(list(stop_words))[:20])

# Apply stop word removal
tokens = ['the', 'fox', 'is', 'running', 'fast', 'through', 'a', 'forest']
filtered = [w for w in tokens if w not in stop_words]

print("\nBefore:", tokens)
print("After: ", filtered)

# NLTK supports 40+ languages!
print("\nAvailable languages (sample):", stopwords.fileids()[:10])

---
## Step 8A — Stemming

**The problem:** "running", "runs", "ran", "runner" are all forms of the same word. For search and retrieval, they should map to one common form.

**Stemming** is the **crude, fast solution** — it chops off suffixes using rules. No dictionary, no intelligence — just pattern-based cutting.

> ⚠️ Stemming often produces words that don't exist in any dictionary.

In [ ]:
from nltk.stem import PorterStemmer, LancasterStemmer, SnowballStemmer

words = ['running', 'studies', 'happiness', 'generous', 'flies', 'easily', 'caring']

# Porter Stemmer — most common, balanced
ps = PorterStemmer()
print("Porter Stemmer:")
for w in words:
    print(f"  {w:15} → {ps.stem(w)}")

print()

# Lancaster Stemmer — most aggressive
ls = LancasterStemmer()
print("Lancaster Stemmer (more aggressive):")
for w in words:
    print(f"  {w:15} → {ls.stem(w)}")

# Notice: Lancaster often over-stems, producing very short roots

In [ ]:
# Snowball Stemmer — multilingual
ss_en = SnowballStemmer('english')
ss_fr = SnowballStemmer('french')

print("Snowball (English):")
for w in words:
    print(f"  {w:15} → {ss_en.stem(w)}")

# Supported languages
print("\nSupported languages:", SnowballStemmer.languages)

---
## Step 8B — Lemmatization

**Lemmatization** is the **smart, accurate solution**. Instead of chopping, it looks up the actual dictionary base form — called the **lemma**.

**Critical trick:** You must tell it the **Part of Speech (POS)** for accurate results.

| POS tag | Meaning |
|---|---|
| `'n'` | Noun (default) |
| `'v'` | Verb |
| `'a'` | Adjective |
| `'r'` | Adverb |

In [ ]:
from nltk.stem import WordNetLemmatizer

lem = WordNetLemmatizer()

# Without POS — assumes NOUN by default
print("Without POS tag (defaults to noun):")
print(f"  running  → {lem.lemmatize('running')}")
print(f"  better   → {lem.lemmatize('better')}")
print(f"  studies  → {lem.lemmatize('studies')}")

print()

# With POS — now it understands grammar
print("With correct POS tag:")
print(f"  running (verb)     → {lem.lemmatize('running', pos='v')}")
print(f"  better (adjective) → {lem.lemmatize('better', pos='a')}")
print(f"  studies (verb)     → {lem.lemmatize('studies', pos='v')}")
print(f"  ran (verb)         → {lem.lemmatize('ran', pos='v')}")
print(f"  easily (adverb)    → {lem.lemmatize('easily', pos='r')}")

# Notice: 'better' → 'good' when POS is adjective! That's real lemmatization.

---
## ⚔️ Stemming vs Lemmatization — Side by Side Comparison

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

ps = PorterStemmer()
lem = WordNetLemmatizer()

test_words = [
    ('studies',   'v'),
    ('running',   'v'),
    ('happiness', 'n'),
    ('better',    'a'),
    ('generous',  'a'),
    ('flies',     'v'),
    ('caring',    'v'),
    ('wolves',    'n'),
]

print(f"{'Word':<15} {'Stemmed':<15} {'Lemmatized':<15} {'Real Word?'}")
print("-" * 60)
for word, pos in test_words:
    stemmed = ps.stem(word)
    lemmatized = lem.lemmatize(word, pos=pos)
    print(f"{word:<15} {stemmed:<15} {lemmatized:<15}")

---
## 🧰 The `re` Module — Complete Guide

`re` is Python's **regular expressions** library. It lets you find, match, and replace **patterns** in text.

### The 4 Main Functions

| Function | What it does |
|---|---|
| `re.sub(pattern, replacement, text)` | Find pattern → replace it |
| `re.findall(pattern, text)` | Find all matches → return as list |
| `re.search(pattern, text)` | Find first match → return match object |
| `re.match(pattern, text)` | Match at START of string only |

In [ ]:
import re

text = "I have 3 cats and 12 dogs and 1 bird"

# re.sub — replace pattern with something
result = re.sub(r'\d+', 'NUM', text)
print("re.sub:    ", result)

# re.findall — get all matches as a list
numbers = re.findall(r'\d+', text)
print("re.findall:", numbers)

# re.search — find first match
match = re.search(r'\d+', text)
print("re.search: ", match.group(), "at position", match.start())

# re.match — only matches at START of string
print("re.match on '3 cats':", re.match(r'\d+', '3 cats'))
print("re.match on 'cats 3':", re.match(r'\d+', 'cats 3'))  # None!

---
### Regex Patterns You Must Know

In [ ]:
text = "Hello World123! Tab:\there. Newline:\nDone."

# Character shortcodes
print("=== Character Shortcodes ===")
print(".  (any char):     ", re.findall(r'.', 'abc!9'))         # every character
print("\\d (digit):        ", re.findall(r'\d', 'abc123def456'))  # digits only
print("\\D (non-digit):    ", re.findall(r'\D', 'abc123'))       # non-digits
print("\\w (word char):    ", re.findall(r'\w', 'hello_123!'))   # letters, digits, _
print("\\W (non-word):     ", re.findall(r'\W', 'hello_123!'))   # symbols, spaces
print("\\s (whitespace):   ", re.findall(r'\s', 'a b\tc\nd'))   # spaces, tabs, newlines
print("\\S (non-whitespace):", re.findall(r'\S', 'a b c'))       # non-spaces

print()
# Quantifiers
print("=== Quantifiers ===")
print("+ (one or more):   ", re.findall(r'\d+', 'abc 123 def 4567'))   # groups digits
print("* (zero or more):  ", re.findall(r'\d*', 'a1b'))                # can be empty
print("? (optional):      ", re.findall(r'colou?r', 'color colour'))   # u is optional
print("{3} (exactly 3):   ", re.findall(r'\d{3}', '12 123 1234 12345'))
print("{2,4} (2 to 4):    ", re.findall(r'\d{2,4}', '1 12 123 1234 12345'))

In [ ]:
# Anchors and character classes
print("=== Anchors ===")
print("^ (start):  ", re.findall(r'^\w+', 'Hello World'))   # first word only
print("$ (end):    ", re.findall(r'\w+$', 'Hello World'))   # last word only

print()
print("=== Character Classes ===")
print("[abc]:         ", re.findall(r'[aeiou]', 'Hello World'))        # vowels
print("[a-z]:         ", re.findall(r'[a-z]+', 'Hello World 123'))    # lowercase words
print("[A-Z]:         ", re.findall(r'[A-Z]+', 'Hello WORLD'))        # uppercase
print("[^aeiou]:      ", re.findall(r'[^aeiou\s]+', 'hello world'))   # consonants

---
### Real Regex Patterns for NLP — The Ones You'll Actually Use

In [ ]:
text = "<p>Visit https://example.com or email hello@test.com — call +91-9876543210 by 3pm!</p>"
print("Original:", text)
print()

# 1. Remove HTML tags
t = re.sub(r'<[^>]+>', '', text)
print("No HTML:   ", t)

# 2. Remove URLs
t = re.sub(r'http\S+|www\S+', '', t)
print("No URLs:   ", t)

# 3. Remove emails
t = re.sub(r'\S+@\S+', '', t)
print("No emails: ", t)

# 4. Remove phone numbers
t = re.sub(r'[\+]?[\d][\d\-\s]{8,}[\d]', '', t)
print("No phones: ", t)

# 5. Remove special characters
t = re.sub(r'[^a-zA-Z0-9\s]', '', t)
print("No special:", t)

# 6. Normalize whitespace
t = re.sub(r'\s+', ' ', t).strip()
print("Normalized:", t)

---
## 🔁 The Full Pipeline — All Steps Combined

Now let's chain everything into one clean function.

In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lem = WordNetLemmatizer()

def preprocess(text, verbose=False):
    steps = []
    
    # Step 1: Lowercase
    text = text.lower()
    if verbose: steps.append(("1. Lowercase", text))
    
    # Step 2: Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    if verbose: steps.append(("2. Remove URLs", text))
    
    # Step 3: Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    if verbose: steps.append(("3. Remove HTML", text))
    
    # Step 4: Remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    if verbose: steps.append(("4. Remove special chars", text))
    
    # Step 5: Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    if verbose: steps.append(("5. Normalize spaces", text))
    
    # Step 6: Tokenize
    tokens = word_tokenize(text)
    if verbose: steps.append(("6. Tokenize", str(tokens)))
    
    # Step 7: Remove stop words
    tokens = [t for t in tokens if t not in stop_words]
    if verbose: steps.append(("7. Remove stop words", str(tokens)))
    
    # Step 8: Lemmatize
    tokens = [lem.lemmatize(t, pos='v') for t in tokens]
    if verbose: steps.append(("8. Lemmatize", str(tokens)))
    
    if verbose:
        for step, result in steps:
            print(f"{step:<25}: {result}")
        print()
    
    return ' '.join(tokens)

# Test it — step by step
raw = "<p>The <b>Quick</b> Brown Foxes are RUNNING swiftly!!! Visit https://nlp.com</p>"
print("INPUT:", raw)
print("=" * 70)
result = preprocess(raw, verbose=True)
print("FINAL OUTPUT:", result)

---
## 🗺️ When to Use What — Decision Guide

| Task | Lowercase | HTML Remove | Stop Words | Stemming | Lemmatization |
|---|---|---|---|---|---|
| TF-IDF / BM25 keyword search | ✅ | ✅ | ✅ | ✅ or ✅ | ✅ |
| Transformer models (BERT/GPT) | ✅ | ✅ | ❌ | ❌ | ❌ |
| Chatbot / RAG pipeline | ✅ | ✅ | ✅ | ❌ | ✅ |
| Sentiment analysis | ✅ | ✅ | ❌ | ❌ | ❌ |
| Named Entity Recognition | ❌ | ✅ | ❌ | ❌ | ❌ |
| Large-scale fast search | ✅ | ✅ | ✅ | ✅ | ❌ |

---
## 📝 Practice Exercises

In [ ]:
# Exercise 1 — Test the pipeline on different inputs
# Try these one by one and observe the output

test_cases = [
    "<p>The <b>Quick</b> Brown Foxes are running swiftly!</p>",
    "Visit https://openai.com for more info about AI!",
    "The studies show that happiness and generous behavior are BETTER outcomes.",
    "   Extra   spaces\t\tand\nnewlines   should   be   cleaned   ",
]

for i, test in enumerate(test_cases, 1):
    print(f"Test {i}:")
    print(f"  Input:  {test}")
    print(f"  Output: {preprocess(test)}")
    print()

In [ ]:
# Exercise 2 — Compare Stemming vs Lemmatization on your own words
# Add any words you want to test

from nltk.stem import PorterStemmer, WordNetLemmatizer

ps = PorterStemmer()
lem = WordNetLemmatizer()

# Try your own words here!
my_words = ['connection', 'computing', 'beautiful', 'worse', 'geese', 'going', 'ate']

print(f"{'Word':<15} {'Stemmed (Porter)':<20} {'Lemma (noun)':<20} {'Lemma (verb)'}")
print("-" * 70)
for w in my_words:
    print(f"{w:<15} {ps.stem(w):<20} {lem.lemmatize(w, 'n'):<20} {lem.lemmatize(w, 'v')}")

In [ ]:
# Exercise 3 — Build a regex pattern from scratch
# Extract all: (a) dates, (b) prices, (c) hashtags from text

text = """
On 12/03/2024, we launched #NLP toolkit. 
Price was $49.99 and $129.00. 
Next update: 01/06/2025. 
Follow #Python and #MachineLearning for updates!
"""

# Extract dates (dd/mm/yyyy)
dates = re.findall(r'\d{2}/\d{2}/\d{4}', text)
print("Dates found:    ", dates)

# Extract prices ($X.XX)
prices = re.findall(r'\$\d+\.\d{2}', text)
print("Prices found:   ", prices)

# Extract hashtags
hashtags = re.findall(r'#\w+', text)
print("Hashtags found: ", hashtags)

---
## ✅ Summary Cheat Sheet

### Preprocessing Steps
```
1. text.lower()                          → Lowercase
2. re.sub(r'http\S+|www\S+', '', text)   → Remove URLs  
3. re.sub(r'<[^>]+>', '', text)          → Remove HTML tags
4. re.sub(r'[^a-zA-Z0-9\s]', '', text)  → Remove special chars
5. re.sub(r'\s+', ' ', text).strip()    → Normalize spaces
6. word_tokenize(text)                   → Tokenize
7. [w for w in tokens if w not in stop_words]  → Stop words
8. lem.lemmatize(word, pos='v')          → Lemmatize
```

### Key Regex Patterns
```
\d     digit          \D    non-digit
\w     word char      \W    non-word char  
\s     whitespace     \S    non-whitespace
+      one or more    *     zero or more
?      optional       {n}   exactly n times
[abc]  any of a,b,c   [^x]  anything except x
^      start          $     end
```

### Stemming vs Lemmatization
```
Stemming:       Fast, crude, may produce non-words  → Use for speed
Lemmatization:  Slow, accurate, always real words   → Use for accuracy
```

---
> 📝 **Next up (Day 5):** These preprocessed, chunked tokens will be converted into **dense vector embeddings** — enabling semantic (meaning-based) search that goes beyond exact keyword matching.